# Lesson 01: Hello Chrono - 자유낙하 시뮬레이션

## 학습 목표
1. `ChSystemNSC`로 물리 세계를 만드는 법
2. `ChBody`로 물체를 만들고 추가하는 법
3. `DoStepDynamics`로 시뮬레이션을 실행하는 법
4. 물체의 위치/속도를 읽는 법

## Chrono API 핵심
| 클래스/메서드 | 역할 |
|---|---|
| `ChSystemNSC()` | Non-Smooth Contact 물리 시스템 생성 |
| `SetGravitationalAcceleration()` | 중력 벡터 설정 |
| `ChBody()` | 강체(rigid body) 생성 |
| `SetMass()`, `SetPos()`, `SetFixed()` | 물체 속성 설정 |
| `AddBody()` | 시스템에 물체 추가 |
| `DoStepDynamics(dt)` | dt초만큼 시뮬레이션 전진 |
| `GetPos()`, `GetPosDt()` | 위치, 속도(위치의 시간미분) 읽기 |

In [ ]:
import pychrono as chrono

## 1단계: 물리 세계(시스템) 만들기

- `ChSystemNSC` = Non-Smooth Contact 방식의 물리 시스템 (빠르고 일반적)
- 대안: `ChSystemSMC` = Smooth Contact (부드러운 접촉, 고무/타이어 등)
- `ChVector3d(x, y, z)` = 3D 벡터. 여기서 y축이 위

In [ ]:
# 물리 세계 생성
sys = chrono.ChSystemNSC()

# 중력 설정: y축 방향으로 -9.81 m/s² (지구 표면)
sys.SetGravitationalAcceleration(chrono.ChVector3d(0, -9.81, 0))

print(f"중력: {sys.GetGravitationalAcceleration().y} m/s²")

## 2단계: 물체(공) 만들기

- `ChBody()` = 강체. 형태가 변하지 않는 단단한 물체
- `SetFixed(False)` → 중력/충돌에 의해 자유롭게 움직임
- `SetFixed(True)` → 바닥처럼 고정 (나중 레슨에서 사용)

In [ ]:
ball = chrono.ChBody()
ball.SetMass(1.0)                           # 질량: 1 kg
ball.SetPos(chrono.ChVector3d(0, 10, 0))    # 초기 위치: 높이 10m
ball.SetFixed(False)                        # 고정하지 않음
sys.AddBody(ball)

print(f"질량: {ball.GetMass()} kg")
print(f"초기 위치: y = {ball.GetPos().y} m")

## 3단계: 시뮬레이션 실행

- `DoStepDynamics(dt)` = 시간을 dt초만큼 전진
- dt가 작을수록 정확하지만 계산 오래 걸림
- `GetChTime()` = 현재 시뮬레이션 시간
- `GetPosDt()` = 속도 (position derivative = 위치의 시간 미분)

In [ ]:
dt = 0.01   # 시간 간격: 10ms

print(f"{'시간(s)':>8}  {'높이(m)':>10}  {'속도(m/s)':>10}  {'이론높이(m)':>12}")
print("─" * 48)

while sys.GetChTime() < 2.0:
    sys.DoStepDynamics(dt)

    t = sys.GetChTime()
    y = ball.GetPos().y          # 현재 높이
    vy = ball.GetPosDt().y       # 현재 y방향 속도

    # 이론값: y = y0 - (1/2) * g * t²
    y_theory = 10.0 - 0.5 * 9.81 * t * t

    # 0.2초마다 출력
    if abs(t % 0.2) < dt * 0.5 or abs(t % 0.2 - 0.2) < dt * 0.5:
        print(f"{t:8.2f}  {y:10.4f}  {vy:10.4f}  {y_theory:12.4f}")

## 4단계: 결과 확인 - 시뮬레이션 vs 이론값

In [ ]:
final_y = ball.GetPos().y
theory_y = 10 - 0.5 * 9.81 * 2.0**2

print(f"시뮬레이션 결과: y = {final_y:.4f} m")
print(f"이론값 (물리):   y = {theory_y:.4f} m")
print(f"오차:            {abs(final_y - theory_y):.6f} m")

## 실험해보기

위 셀들을 수정하며 다음을 확인해보세요:

1. **dt를 바꾸면?** `dt = 0.1` vs `dt = 0.001` → 오차 변화 관찰
2. **질량을 바꾸면?** `SetMass(100)` → 자유낙하는 질량과 무관 (갈릴레오)
3. **중력을 바꾸면?** `ChVector3d(0, -1.62, 0)` → 달 표면 (1.62 m/s²)
4. **초기 높이를 바꾸면?** `ChVector3d(0, 100, 0)` → 더 긴 낙하